# 00b · EverydayScents: label cleaning

**Purpose.** Read every session metadata file, map raw labels to the 4 classes (ambient, coffee, truffle, black_pepper) and drop test / unrelated samples. Check class counts per session and per day. Open item: confirm whether `old_dataset.csv` and `new_truffle_dataset.csv` overlap with the session files.

**Inputs**
- `data/raw/everyday_scents/session_<id>_metadata.json`: 81 sessions (samples, labels, phase durations, gains)

**Outputs**
- `data/processed/everyday_scents/meta.csv`: 155 rows (session_id, sample, day, raw label, class, y_multi with counts 37 / 42 / 54 / 22, y_binary = truffle vs not truffle with 54 / 101)

In [1]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src import load_es

RAW = ROOT / "data/raw/everyday_scents"
OUT = ROOT / "data/processed/everyday_scents"
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
samples = load_es.load_samples(RAW)
print(samples["session_id"].nunique(), "sessions,", len(samples), "samples")
samples["raw_label"].value_counts()

73 sessions, 172 samples


raw_label
ambient                             37
truffle                             29
coffee                              28
fresh_truffle                       20
black_pepper                        19
test                                 6
Arabica_coffee                       4
truffle_outdoor                      4
black_paper                          3
coffee_outdoor                       3
Robusta_coffee                       3
test_fan_truffle                     2
test_V5_coffee_fan                   2
coffee_ground_heated                 2
coffee ground                        2
burned_soil                          1
forest_soil                          1
a                                    1
test_coffee_fan                      1
test_coffee_fan_diff                 1
test_coffee_fan_diff_closedLid       1
test_coffee_fan_noDiff_closedLid     1
fresh_truffle_with_jar               1
Name: count, dtype: int64

## Label map: raw label → class

In [3]:
pd.crosstab(samples["raw_label"], samples["class"].fillna("(dropped)"))

class,(dropped),ambient,black_pepper,coffee,truffle
raw_label,,,,,
Arabica_coffee,0,0,0,4,0
Robusta_coffee,0,0,0,3,0
a,1,0,0,0,0
ambient,0,37,0,0,0
black_paper,0,0,3,0,0
black_pepper,0,0,19,0,0
burned_soil,1,0,0,0,0
coffee,0,0,0,28,0
coffee ground,0,0,0,2,0


In [4]:
dropped = samples[samples["class"].isna()]
print(len(dropped), "dropped:", sorted(dropped["raw_label"].unique()))

meta = samples.dropna(subset=["class"]).reset_index(drop=True)
meta["y_multi"] = meta["class"].map({c: i for i, c in enumerate(load_es.CLASSES)})
meta["y_binary"] = (meta["y_multi"] == load_es.TRUFFLE_CLASS).astype(int)

counts = meta["class"].value_counts()
print(counts)
assert len(meta) == 155 and len(dropped) == 17
assert counts.to_dict() == {"ambient": 37, "coffee": 42, "truffle": 54, "black_pepper": 22}

17 dropped: ['a', 'burned_soil', 'forest_soil', 'test', 'test_V5_coffee_fan', 'test_coffee_fan', 'test_coffee_fan_diff', 'test_coffee_fan_diff_closedLid', 'test_coffee_fan_noDiff_closedLid', 'test_fan_truffle']
class
truffle         54
coffee          42
ambient         37
black_pepper    22
Name: count, dtype: int64


## Sessions and days per class
Folds are grouped by session, so a class with few sessions (black_pepper) is the hardest to split.

In [5]:
meta.groupby("class").agg(samples=("sample_id", "size"), sessions=("session_id", "nunique"),
                          days=("day", "nunique")).loc[load_es.CLASSES]

,samples,sessions,days
class,,,
ambient,37,14,9
coffee,42,20,11
truffle,54,20,10
black_pepper,22,8,4


In [6]:
pd.crosstab(meta["day"], meta["class"])

class,ambient,black_pepper,coffee,truffle
day,,,,
20260703,0,0,0,1
20260707,0,0,1,6
20260708,0,0,0,15
20260714,3,3,3,5
20260722,3,0,3,3
20260723,0,0,3,0
20260727,6,0,6,7
20260728,3,6,3,0
20260729,4,7,9,6


## Open item: do `old_dataset.csv` and `new_truffle_dataset.csv` overlap with the sessions?
Compared by time range. The plan uses the session files only; this is a report, nothing is merged.

In [7]:
sess_ranges = []
for sid in samples["session_id"].unique():
    ts = pd.to_datetime(pd.read_csv(RAW / f"{sid}.csv", usecols=["timestamp"])["timestamp"])
    sess_ranges.append((sid, ts.min(), ts.max()))
sess_ranges = pd.DataFrame(sess_ranges, columns=["session_id", "start", "end"])

for name in ["old_dataset.csv", "new_truffle_dataset.csv"]:
    ts = pd.to_datetime(pd.read_csv(RAW / name, usecols=["timestamp"])["timestamp"])
    overlap = sess_ranges[(sess_ranges["start"] <= ts.max()) & (sess_ranges["end"] >= ts.min())]
    print(f"{name}: {ts.min()} -> {ts.max()}, overlapping sessions: {list(overlap['session_id'])}")
print("sessions:", sess_ranges["start"].min(), "->", sess_ranges["end"].max())

old_dataset.csv: 2026-06-23 13:17:27.179000 -> 2026-06-23 13:34:37.730000, overlapping sessions: []
new_truffle_dataset.csv: 2026-06-25 09:25:34.442000 -> 2026-06-25 11:21:55.582000, overlapping sessions: []
sessions: 2026-07-03 13:40:00.184000 -> 2026-08-20 10:02:32.737000


In [8]:
meta.to_csv(OUT / "meta.csv", index=False)
print("saved", len(meta), "rows to", OUT / "meta.csv")

saved 155 rows to ..\..\data\processed\everyday_scents\meta.csv
